# Cell 1: Imports & Setup

In [ ]:
import os
import json
import torch
import torch.nn as nn
import pandas as pd
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from dataclasses import dataclass

# Gemini Imports
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

from dotenv import load_dotenv
load_dotenv()

# 1. Mount Drive

# 2. Authentication

# Initialize Gemini Client
gemini_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# 3. Hardware check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Cell 2: Configuration

In [ ]:
@dataclass
class InjectorConfig:
    # --- Paths ---
    base_dir: str = "."

    # Phase 1 Paths
    train_csv: str = "./RepliQA_Train.csv"
    train_vectors_dir: str = "./RepliQA_Train_vectors"
    save_path_phase1: str = "./injector_V3_phase1_SFT.pt"

    # Phase 2 Paths (RL)
    val_csv: str = "./RepliQA_Val.csv"
    phase2_vectors_dir: str = "./RepliQA_Test_vectors"
    save_path_phase2: str = "./injector_V3_phase2_RL.pt"

    # --- Model Specs ---
    model_name: str = "meta-llama/Llama-3.1-8B-Instruct"
    torch_dtype: torch.dtype = torch.bfloat16
    injection_layer: int = 16

    # --- Injector Architecture (V2) ---
    original_dim: int = 4096
    # max_context_vectors: int = 93

    # --- Training Params ---
    phase_one_epochs: int = 100
    phase_two_epochs: int = 10
    batch_size: int = 4
    lr_phase1: float = 1e-3
    lr_phase2: float = 1e-4

    # RL Params
    reward_weights = {"correctness": 0.6, "similarity": 0.2, "adherence": 0.2}
    baseline: float = 0.5

cfg = InjectorConfig()
print("Two-Phase Configuration Loaded.")

Two-Phase Configuration Loaded.


# Cell 3: Dataset & Data Loading

In [ ]:
class RepliQADataset(Dataset):
    def __init__(self, df, vectors_dir, tokenizer, cfg):
        self.data = df.to_dict('records')
        self.vectors_dir = vectors_dir
        self.tokenizer = tokenizer
        self.cfg = cfg
        self.available_files = set(os.listdir(vectors_dir))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        question = str(row['question'])
        answer = str(row['answer'])
        q_id = str(row['question_id'])

        # Safely grab the text context for the Judge (Check common column names)
        context_text = str(row.get('long_answer', row.get('context', '')))

        safe_id = "".join([c for c in q_id if c.isalnum() or c in ('-','_')])
        filename = f"vec_{safe_id}.pt"

        if filename in self.available_files:
            file_path = os.path.join(self.vectors_dir, filename)
            try:
                loaded_data = torch.load(file_path, map_location="cpu", weights_only=False)
                context_vectors = loaded_data['vectors']
            except Exception as e:
                context_vectors = torch.zeros((1, self.cfg.original_dim))
        else:
            context_vectors = torch.zeros((1, self.cfg.original_dim))

        prompt_text = (
            "Answer the question based strictly on the context below.\n"
            f"Question: {question}\n"
            "<context> :"
        )

        return {
            "prompt_text": prompt_text,
            "answer_text": answer,
            "context_vectors": context_vectors,
            "context_text": context_text
        }

def collate_fn(batch):
    return batch

# Cell 4: The Injector Model Architecture

In [ ]:
import torch
import torch.nn as nn

class TriContextInjector(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed_dim = cfg.original_dim

        # 1. Learned positional weights (assuming context won't exceed 150 tokens)
        self.pos_weights = nn.Parameter(torch.ones(150, 1))

        # 2. A lightweight 1-layer Transformer to mix the 4 vectors
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.embed_dim,
            nhead=8,
            dim_feedforward=self.embed_dim * 2,
            batch_first=True,
            norm_first=True # More stable for RL training
        )
        self.mixer = nn.TransformerEncoder(encoder_layer, num_layers=1)

        # 3. Final projection for the residual connection
        self.final_proj = nn.Linear(self.embed_dim, self.embed_dim)

        # Zero-initialize the final layer so Step 0 acts exactly like frozen Llama
        nn.init.zeros_(self.final_proj.weight)
        nn.init.zeros_(self.final_proj.bias)

    def forward(self, target_token_vec, context_vecs_list):
        device = target_token_vec.device
        dtype = target_token_vec.dtype

        mixed_outputs = []

        # Process each sequence to extract the 3 feature vectors
        for i, vec in enumerate(context_vecs_list):
            vec = vec.to(device).to(dtype)
            seq_len = vec.shape[0]

            # --- Vector 1: The Last Vector ---
            # Contains causal attention summary of the preceding tokens
            v_last = vec[-1, :].unsqueeze(0) # Shape: [1, 4096]

            # --- Vector 2: The Mean Vector ---
            # Captures global semantics without positional bias
            v_mean = vec.mean(dim=0, keepdim=True) # Shape: [1, 4096]

            # --- Vector 3: The Weighted Positional Mean ---
            # Applies a learned softmax weight to the sequence length
            w = torch.softmax(self.pos_weights[:seq_len], dim=0)
            v_weighted = (vec * w).sum(dim=0, keepdim=True) # Shape: [1, 4096]

            # --- Vector 4: The Target Vector ---
            v_target = target_token_vec[i] # Shape: [1, 4096]

            # Stack into a fixed 4-token sequence: [Target, Last, Mean, Weighted]
            seq_4 = torch.cat([v_target, v_last, v_mean, v_weighted], dim=0) # [4, 4096]
            mixed_outputs.append(seq_4)

        # Batch the 4-token sequences: [B, 4, 4096]
        batch_seq_4 = torch.stack(mixed_outputs, dim=0)

        # Let the Transformer find the relationships between these 4 vectors
        transformer_out = self.mixer(batch_seq_4) # [B, 4, 4096]

        # Extract only the output corresponding to our target vector (Index 0)
        target_out = transformer_out[:, 0, :] # [B, 4096]

        # Apply final projection and Residual Addition
        final_vec = target_token_vec[:, 0, :] + self.final_proj(target_out)

        return final_vec.unsqueeze(1) # [B, 1, 4096]

# Cell 5: Training Logic (The Hook)

## Phase 1

In [ ]:
# --- PHASE 1: Supervised Fine Tuning ---
def train_phase1_epoch(model, injector, dataloader, optimizer, cfg, tokenizer):
    model.eval()
    injector.train()
    total_loss = 0
    loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    for step, batch in enumerate(tqdm(dataloader, desc="Phase 1 (SFT) Training")):
        optimizer.zero_grad()
        prompts = [item['prompt_text'] for item in batch]
        answers = [item['answer_text'] + tokenizer.eos_token for item in batch]
        context_vecs = [item['context_vectors'] for item in batch]

        p_enc = tokenizer(prompts, return_length=True, add_special_tokens=True)
        a_enc = tokenizer(answers, add_special_tokens=False)

        input_ids_list, labels_list, injection_indices = [], [], []

        for i in range(len(prompts)):
            p_ids = p_enc['input_ids'][i]
            a_ids = a_enc['input_ids'][i]
            full_ids = p_ids + a_ids
            labels = [-100] * len(p_ids) + a_ids

            input_ids_list.append(torch.tensor(full_ids))
            labels_list.append(torch.tensor(labels))
            injection_indices.append(len(p_ids) - 1)

        input_ids = torch.nn.utils.rnn.pad_sequence(input_ids_list, batch_first=True, padding_value=tokenizer.pad_token_id).to(device)
        labels = torch.nn.utils.rnn.pad_sequence(labels_list, batch_first=True, padding_value=-100).to(device)
        attention_mask = (input_ids != tokenizer.pad_token_id).long().to(device)

        def forward_hook(module, args, output):
            hs = output[0] if isinstance(output, tuple) else output
            target_vectors = [hs[b_idx, seq_idx, :].unsqueeze(0).unsqueeze(0) for b_idx, seq_idx in enumerate(injection_indices)]
            target_batch = torch.cat(target_vectors, dim=0)

            replacement_batch = injector(target_batch, context_vecs)
            hs_modified = hs.clone()

            for b_idx, seq_idx in enumerate(injection_indices):
                hs_modified[b_idx, seq_idx, :] = replacement_batch[b_idx, 0, :]
            return (hs_modified,) + output[1:] if isinstance(output, tuple) else hs_modified

        hook_handle = model.model.layers[cfg.injection_layer].register_forward_hook(forward_hook)

        try:
            with torch.autocast(device_type='cuda', dtype=cfg.torch_dtype):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        finally:
            hook_handle.remove()

    return total_loss / len(dataloader)


## Phase 2

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Loading Local Qwen-3B Judge...")
judge_model_id = "Qwen/Qwen2.5-3B-Instruct"

judge_tokenizer = AutoTokenizer.from_pretrained(judge_model_id)
judge_tokenizer.pad_token = judge_tokenizer.eos_token

judge_model = AutoModelForCausalLM.from_pretrained(
    judge_model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
judge_model.eval()
for param in judge_model.parameters():
    param.requires_grad = False

print("Local Judge Loaded Successfully!")

Loading Local Qwen-3B Judge...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Local Judge Loaded Successfully!


In [ ]:
# --- PHASE 2: Qwen Judge & RLHF ---
import json
import re

class EvaluationScores(BaseModel):
    similarity_check: float = Field(description="Score between 0.0 and 1.0")
    correctness_check: float = Field(description="Score between 0.0 and 1.0")
    context_adherence: float = Field(description="Score between 0.0 and 1.0")


def local_qwen_judge(question, context, gold_answer, generated_answer, cfg):
    prompt = f"""You are an expert evaluator for RAG systems.
Analyze the following data points and return a JSON object.

**Input Data:**
- Question: "{question}"
- Context: "{context}"
- Gold Answer: "{gold_answer}"
- Generated Answer: "{generated_answer}"

**Task:**
Evaluate the 'Generated Answer' on these 3 metrics using a float scale from 0.0 to 1.0 (where 0.0 is completely wrong/absent, and 1.0 is perfect):
1. similarity_check: Score the semantic similarity between the Generated Answer and the Gold Answer.
2. correctness_check: Score how correctly the Generated Answer answers the Question.
3. context_adherence: Score how well the Generated Answer relies ONLY on data/information derived from the provided Context.

**Output Format:**
Return ONLY a JSON object with keys: "similarity_check", "correctness_check", "context_adherence".
"""

    messages = [
        {"role": "system", "content": "You are a strict JSON output machine. You only output valid JSON."},
        {"role": "user", "content": prompt}
    ]

    text = judge_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = judge_tokenizer(text, return_tensors="pt").to(judge_model.device)

    with torch.no_grad():
        outputs = judge_model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.0,    # Greedy decoding for consistent grading
            do_sample=False,
            pad_token_id=judge_tokenizer.eos_token_id
        )

    response_text = judge_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    # Robust JSON extraction
    try:
        json_match = re.search(r'\{.*?\}', response_text, re.DOTALL)
        if json_match:
            data = json.loads(json_match.group(0))
        else:
            data = json.loads(response_text)

        c_check = max(0.0, min(1.0, float(data.get('correctness_check', 0.0))))
        s_check = max(0.0, min(1.0, float(data.get('similarity_check', 0.0))))
        a_check = max(0.0, min(1.0, float(data.get('context_adherence', 0.0))))

    except Exception as e:
        print(f"JSON Parsing error: {e} | Raw Output: {response_text}")
        c_check, s_check, a_check = 0.0, 0.0, 0.0

    return (cfg.reward_weights["correctness"] * c_check +
            cfg.reward_weights["similarity"] * s_check +
            cfg.reward_weights["adherence"] * a_check)


In [ ]:
import torch
from tqdm import tqdm

# --- PHASE 2: GRPO RLHF ---
GROUP_SIZE = 4

def train_phase2_epoch(model, injector, dataloader, optimizer, cfg, tokenizer):
    model.eval()
    injector.train()
    total_loss, total_reward, valid_steps = 0, 0, 0
    device = next(model.parameters()).device

    for step, batch in enumerate(tqdm(dataloader, desc="Phase 2 (GRPO) Training")):
        for item in batch:
            prompt = item['prompt_text']
            gold_answer = item['answer_text']
            context_text = item['context_text']
            context_vecs = [item['context_vectors']]

            inputs = tokenizer(prompt, return_tensors="pt").to(device)
            injection_idx = inputs['input_ids'].shape[1] - 1

            # ==========================================
            # 1. GROUP ROLLOUT
            # ==========================================
            def rollout_hook(module, args, output):
                hs = output[0] if isinstance(output, tuple) else output
                # Only inject during the initial prompt processing, not during autoregressive decoding
                if hs.shape[1] > 1:
                    target_vec = hs[:, injection_idx, :].unsqueeze(1)
                    with torch.no_grad():
                        replacement = injector(target_vec, context_vecs)
                    hs[:, injection_idx, :] = replacement[:, 0, :]
                return (hs,) + output[1:] if isinstance(output, tuple) else hs

            handle = model.model.layers[cfg.injection_layer].register_forward_hook(rollout_hook)

            with torch.no_grad():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=80,
                    do_sample=True,
                    temperature=0.8,
                    num_return_sequences=GROUP_SIZE,
                    pad_token_id=tokenizer.eos_token_id
                )
            handle.remove()

            # ==========================================
            # 2. GROUP EVALUATION (Using Local Qwen Judge)
            # ==========================================
            question = prompt.split('Question: ')[1].split('<context>')[0].strip()
            group_rewards = []

            for i in range(GROUP_SIZE):
                generated_answer = tokenizer.decode(output_ids[i][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
                reward = local_qwen_judge(question, context_text, gold_answer, generated_answer, cfg)
                group_rewards.append(reward)

            rewards_tensor = torch.tensor(group_rewards, dtype=torch.float32, device=device)
            mean_reward = rewards_tensor.mean()
            std_reward = rewards_tensor.std() + 1e-8

            advantages = (rewards_tensor - mean_reward) / std_reward
            total_reward += mean_reward.item()

            # ==========================================
            # 3. GROUP UPDATE
            # ==========================================
            optimizer.zero_grad()
            step_loss = 0

            def update_hook(module, args, output):
                hs = output[0] if isinstance(output, tuple) else output
                target_vec = hs[:, injection_idx, :].unsqueeze(1)
                replacement = injector(target_vec, context_vecs)
                hs_mod = hs.clone()
                hs_mod[:, injection_idx, :] = replacement[:, 0, :]
                return (hs_mod,) + output[1:] if isinstance(output, tuple) else hs_mod

            handle = model.model.layers[cfg.injection_layer].register_forward_hook(update_hook)

            try:
                for i in range(GROUP_SIZE):
                    single_output_ids = output_ids[i:i+1]
                    labels = single_output_ids.clone()
                    labels[0, :inputs['input_ids'].shape[1]] = -100

                    with torch.autocast(device_type='cuda', dtype=cfg.torch_dtype):
                        outputs = model(input_ids=single_output_ids, labels=labels)
                        ce_loss = outputs.loss

                    policy_loss = (ce_loss * advantages[i]) / GROUP_SIZE
                    policy_loss.backward()
                    step_loss += policy_loss.item()

                optimizer.step()
                total_loss += step_loss
                valid_steps += 1

            finally:
                handle.remove()

    return (total_loss / valid_steps if valid_steps > 0 else 0), (total_reward / valid_steps if valid_steps > 0 else 0)

# Cell 6: Main Execution

## Phase 1

In [ ]:
# Helper to prevent crashes from old architecture weights
def safe_load_checkpoint(model, path):
    if os.path.exists(path):
        try:
            state_dict = torch.load(path, map_location=device)
            model.load_state_dict(state_dict)
            print(f"✅ Successfully loaded weights from {path}")
            return True
        except Exception as e:
            print(f"⚠️ Architecture mismatch detected at {path}.")
            print(f"   Error: {str(e)[:100]}...")
            print("   Discarding incompatible weights and starting fresh.")
            return False
    return False

In [ ]:
# 1. Load Main Model
print(f"Loading {cfg.model_name}...")
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    device_map="auto",
    torch_dtype=cfg.torch_dtype,
    attn_implementation="sdpa"
)
model.eval()
for param in model.parameters():
    param.requires_grad = False

# Instantiate the NEW Architecture
injector = TriContextInjector(cfg).to(device, dtype=cfg.torch_dtype)

Loading meta-llama/Llama-3.1-8B-Instruct...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

/tmp/ipykernel_845/1104558716.py:21: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.mixer = nn.TransformerEncoder(encoder_layer, num_layers=1)


In [ ]:
# ==========================================
# PHASE 1: Supervised Fine-Tuning
# ==========================================
print("\n=== Preparing Phase 1 (SFT) ===")

# Check for checkpoint, but do NOT skip training
if safe_load_checkpoint(injector, cfg.save_path_phase1):
    print("Resuming SFT training from the loaded checkpoint...")
else:
    print("Starting Supervised Fine-Tuning from scratch...")

# ALWAYS PREPARE DATA AND TRAIN PHASE 1
df_train = pd.read_csv(cfg.train_csv)
dataset_p1 = RepliQADataset(df_train, cfg.train_vectors_dir, tokenizer, cfg)
dataloader_p1 = DataLoader(dataset_p1, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn, num_workers=0, pin_memory=True)

# We train all parameters in Phase 1
optimizer_p1 = torch.optim.AdamW(injector.parameters(), lr=cfg.lr_phase1)

print(f"Running Phase 1 Training for {cfg.phase_one_epochs} epochs...")
for epoch in range(cfg.phase_one_epochs):
    avg_loss = train_phase1_epoch(model, injector, dataloader_p1, optimizer_p1, cfg, tokenizer)
    print(f"P1 Epoch {epoch+1}/{cfg.phase_one_epochs} | Loss: {avg_loss:.4f}")

    # Save checkpoint after every epoch
    torch.save(injector.state_dict(), cfg.save_path_phase1)

print("Phase 1 SFT Complete. Weights saved.")


=== Preparing Phase 1 (SFT) ===
Starting Supervised Fine-Tuning from scratch...
Running Phase 1 Training for 100 epochs...


Phase 1 (SFT) Training:   0%|          | 0/5362 [00:00<?, ?it/s]

P1 Epoch 1/100 | Loss: 2.2345


Phase 1 (SFT) Training:   0%|          | 0/5362 [00:00<?, ?it/s]

P1 Epoch 2/100 | Loss: 2.1410


...

P1 Epoch 98/100 | Loss: 0.1106


Phase 1 (SFT) Training:   0%|          | 0/5362 [00:00<?, ?it/s]

P1 Epoch 99/100 | Loss: 0.1128


Phase 1 (SFT) Training:   0%|          | 0/5362 [00:00<?, ?it/s]

P1 Epoch 100/100 | Loss: 0.1074
Phase 1 SFT Complete. Weights saved.


## Phase 2

In [ ]:
# ==========================================
# PHASE 2: Reinforcement Learning (GRPO)
# ==========================================
import pandas as pd
import torch
from torch.utils.data import DataLoader

print("\n=== Preparing Phase 2 (GRPO with Local Qwen Judge) ===")

# Try to load Phase 2 weights first. If not, fallback to Phase 1 baseline.
if safe_load_checkpoint(injector, cfg.save_path_phase2):
    print("Resuming RL training from the loaded Phase 2 checkpoint...")
elif safe_load_checkpoint(injector, cfg.save_path_phase1):
    print("Starting RL training using the Phase 1 baseline weights...")
else:
    print("⚠️ No compatible weights found. Starting RL from random initialization...")

df_val = pd.read_csv(cfg.val_csv)
dataset_p2 = RepliQADataset(df_val, cfg.phase2_vectors_dir, tokenizer, cfg)

# Added pin_memory=True for faster CPU-to-GPU data transfer
dataloader_p2 = DataLoader(
    dataset_p2,
    batch_size=cfg.batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

# 1. Freeze the Positional Weights to retain the SFT baseline knowledge
injector.pos_weights.requires_grad = False

# 2. Collect only the TriContext Transformer and the Final Projection layers for RL updates
rl_trainable_params = list(injector.mixer.parameters()) + list(injector.final_proj.parameters())

# 3. Initialize the Optimizer
optimizer_p2 = torch.optim.AdamW(rl_trainable_params, lr=cfg.lr_phase2)

print(f"Running Phase 2 Training for {cfg.phase_two_epochs} epochs...")
for epoch in range(cfg.phase_two_epochs):
    avg_loss, avg_reward = train_phase2_epoch(model, injector, dataloader_p2, optimizer_p2, cfg, tokenizer)
    print(f"P2 Epoch {epoch+1}/{cfg.phase_two_epochs} | Policy Loss: {avg_loss:.4f} | Avg Reward: {avg_reward:.4f}")

    # Save checkpoint after every epoch
    torch.save(injector.state_dict(), cfg.save_path_phase2)

print("\nPipeline Complete!")


=== Preparing Phase 2 (GRPO with Local Qwen Judge) ===
✅ Successfully loaded weights from ./injector_V3_phase1_SFT.pt
Starting RL training using the Phase 1 baseline weights...
Running Phase 2 Training for 10 epochs...




Phase 2 (GRPO) Training:   0%|          | 0/298 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Phase 2 (GRPO) Training:   0%|          | 1/298 [00:36<3:00:35, 36.48s/it]

Phase 2 (GRPO) Training:   1%|          | 2/298 [01:10<2:52:31, 34.97s/it]

Phase 2 (GRPO) Training:   1%|          | 3/298 [01:45<2:52:59, 35.18s/it]

Phase 2 (GRPO) Training:   1%|▏         | 4/298 [02:24<2:59:01, 36.53s/it]

Phase 2 (GRPO) Training:   2%|▏         | 5/298 [03:00<2:56:48, 36.21s/it]

Phase 2 (GRPO) Training:   2%|▏         | 6/298 [03:36<2:56:28, 36.26s/it]

Phase 2 (GRPO) Training:   2%|▏         | 7/298 [04:09<2:50:13, 35.10s/it]

Phase 2 (GRPO) Training:   3%|▎         | 8/298 [04:42<2:47:13, 34.60s/it]

Phase 2 (GRPO) Training:   3%|▎         | 9/298 [05:19<2:50:26, 35.39s/it]

Phase 2 (GRPO) Training:   3%|▎         | 10/298 [05:55<2:49:47, 35.37s/it]

Phase 2 (GRPO) Trai

P2 Epoch 1/10 | Policy Loss: -0.0838 | Avg Reward: 0.3442




Phase 2 (GRPO) Training:   0%|          | 0/298 [00:00<?, ?it/s]

Phase 2 (GRPO) Training:   0%|          | 1/298 [00:31<2:38:02, 31.93s/it]

Phase 2 (GRPO) Training:   1%|          | 2/298 [01:03<2:35:58, 31.62s/it]

Phase 2 (GRPO) Training:   1%|          | 3/298 [01:33<2:32:00, 30.92s/it]

Phase 2 (GRPO) Training:   1%|▏         | 4/298 [02:03<2:30:46, 30.77s/it]

Phase 2 (GRPO) Training:   2%|▏         | 5/298 [02:36<2:33:14, 31.38s/it]

Phase 2 (GRPO) Training:   2%|▏         | 6/298 [03:06<2:31:20, 31.10s/it]

Phase 2 (GRPO) Training:   2%|▏         | 7/298 [03:38<2:31:44, 31.29s/it]

Phase 2 (GRPO) Training:   3%|▎         | 8/298 [04:10<2:31:31, 31.35s/it]

Phase 2 (GRPO) Training:   3%|▎         | 9/298 [04:42<2:33:02, 31.77s/it]

Phase 2 (GRPO) Training:   3%|▎         | 10/298 [05:14<2:32:31, 31.78s/it]

Phase 2 (GRPO) Training:   4%|▎         | 11/298 [05:45<2:30:15, 31.41s/it]

Phase 2 (GRPO) Training:   4%|▍         | 12/298 [06:15<2:27:42, 30.99s/it]

Phase 2 (GRPO) Tr

P2 Epoch 2/10 | Policy Loss: -0.1060 | Avg Reward: 0.3636


...



Phase 2 (GRPO) Training:   0%|          | 0/298 [00:00<?, ?it/s]

Phase 2 (GRPO) Training:   0%|          | 1/298 [00:33<2:47:39, 33.87s/it]

Phase 2 (GRPO) Training:   1%|          | 2/298 [01:09<2:50:48, 34.62s/it]

Phase 2 (GRPO) Training:   1%|          | 3/298 [01:43<2:49:08, 34.40s/it]

Phase 2 (GRPO) Training:   1%|▏         | 4/298 [02:18<2:49:58, 34.69s/it]

Phase 2 (GRPO) Training:   2%|▏         | 5/298 [02:52<2:47:59, 34.40s/it]

Phase 2 (GRPO) Training:   2%|▏         | 6/298 [03:25<2:45:19, 33.97s/it]

Phase 2 (GRPO) Training:   2%|▏         | 7/298 [03:57<2:42:07, 33.43s/it]

Phase 2 (GRPO) Training:   3%|▎         | 8/298 [04:29<2:39:22, 32.97s/it]

Phase 2 (GRPO) Training:   3%|▎         | 9/298 [05:03<2:39:26, 33.10s/it]

Phase 2 (GRPO) Training:   3%|▎         | 10/298 [05:33<2:34:38, 32.22s/it]

Phase 2 (GRPO) Training:   4%|▎         | 11/298 [06:06<2:36:02, 32.62s/it]

Phase 2 (GRPO) Training:   4%|▍         | 12/298 [06:37<2:32:25, 31.98s/it]

Phase 2 (GRPO) Tr

P2 Epoch 7/10 | Policy Loss: -0.1102 | Avg Reward: 0.3881




Phase 2 (GRPO) Training:   0%|          | 0/298 [00:00<?, ?it/s]

Phase 2 (GRPO) Training:   0%|          | 1/298 [00:36<3:03:06, 36.99s/it]

Phase 2 (GRPO) Training:   1%|          | 2/298 [01:11<2:56:04, 35.69s/it]

Phase 2 (GRPO) Training:   1%|          | 3/298 [01:45<2:51:54, 34.96s/it]

Phase 2 (GRPO) Training:   1%|▏         | 4/298 [02:17<2:45:15, 33.73s/it]

Phase 2 (GRPO) Training:   2%|▏         | 5/298 [02:49<2:41:10, 33.00s/it]

Phase 2 (GRPO) Training:   2%|▏         | 6/298 [03:22<2:40:03, 32.89s/it]

Phase 2 (GRPO) Training:   2%|▏         | 7/298 [03:56<2:42:10, 33.44s/it]

Phase 2 (GRPO) Training:   3%|▎         | 8/298 [04:25<2:35:00, 32.07s/it]

Phase 2 (GRPO) Training:   3%|▎         | 9/298 [04:56<2:33:06, 31.79s/it]

Phase 2 (GRPO) Training:   3%|▎         | 10/298 [05:30<2:35:41, 32.44s/it]

Phase 2 (GRPO) Training:   4%|▎         | 11/298 [06:03<2:35:20, 32.48s/it]

Phase 2 (GRPO) Training:   4%|▍         | 12/298 [06:37<2:36:53, 32.91s/it]

Phase 2 (GRPO) Tr

P2 Epoch 8/10 | Policy Loss: -0.0495 | Avg Reward: 0.3859




Phase 2 (GRPO) Training:   0%|          | 0/298 [00:00<?, ?it/s]

Phase 2 (GRPO) Training:   0%|          | 1/298 [00:31<2:35:29, 31.41s/it]

Phase 2 (GRPO) Training:   1%|          | 2/298 [01:02<2:33:30, 31.12s/it]

Phase 2 (GRPO) Training:   1%|          | 3/298 [01:41<2:51:12, 34.82s/it]

Phase 2 (GRPO) Training:   1%|▏         | 4/298 [02:15<2:48:11, 34.33s/it]

Phase 2 (GRPO) Training:   2%|▏         | 5/298 [02:46<2:42:45, 33.33s/it]

Phase 2 (GRPO) Training:   2%|▏         | 6/298 [03:20<2:42:42, 33.43s/it]

Phase 2 (GRPO) Training:   2%|▏         | 7/298 [03:56<2:46:12, 34.27s/it]

Phase 2 (GRPO) Training:   3%|▎         | 8/298 [04:31<2:47:29, 34.65s/it]

Phase 2 (GRPO) Training:   3%|▎         | 9/298 [05:05<2:45:39, 34.39s/it]

Phase 2 (GRPO) Training:   3%|▎         | 10/298 [05:40<2:46:02, 34.59s/it]

Phase 2 (GRPO) Training:   4%|▎         | 11/298 [06:15<2:45:12, 34.54s/it]

Phase 2 (GRPO) Training:   4%|▍         | 12/298 [06:48<2:43:24, 34.28s/it]

Phase 2 (GRPO) Tr

In [ ]:
safe_load_checkpoint(injector, cfg.save_path_phase2)

✅ Successfully loaded weights from ./injector_V3_phase2_RL.pt


True

# Cell 7: Qualitative Evaluation (Generate 5 Samples)

In [ ]:
import random
import pandas as pd
import os

# Path to save the qualitative training samples
TRAIN_RESULTS_SAVE_PATH = os.path.join(cfg.base_dir, "train_qualitative_samples-V3.csv")

def evaluate_random_samples(model, tokenizer, injector, dataset, num_samples=5):
    print(f"\n--- Generating {num_samples} Random Samples from Training Set ---\n")

    # 1. Pick random indices
    indices = random.sample(range(len(dataset)), num_samples)

    model.eval()
    injector.eval()

    results = []

    for idx in indices:
        item = dataset[idx]
        prompt = item['prompt_text']
        true_answer = item['answer_text']

        # Retrieve the raw long_answer (The injected context sentence)
        injected_sentence = dataset.data[idx]['long_answer']

        # Load the context vectors (Sentence)
        context_vecs = [item['context_vectors'].to(device).to(cfg.torch_dtype)]

        # Tokenize Prompt
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        input_ids = inputs['input_ids']

        # Injection Index: Last token of prompt (the colon)
        injection_index = input_ids.shape[1] - 1

        # --- Define Inference Hook ---
        def inference_hook(module, args, output):
            if isinstance(output, tuple):
                hs = output[0]
            else:
                hs = output

            # Inject only during prompt processing
            if hs.shape[1] > 1:
                target_vec = hs[:, injection_index, :].unsqueeze(1)

                with torch.no_grad():
                    replacement = injector(target_vec, context_vecs)

                hs[:, injection_index, :] = replacement[:, 0, :]

            if isinstance(output, tuple):
                return (hs,) + output[1:]
            return hs

        # Register Hook
        handle = model.model.layers[cfg.injection_layer].register_forward_hook(inference_hook)

        try:
            # Generate
            output_ids = model.generate(
                **inputs,
                max_new_tokens=80,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

            # Decode
            generated_text = tokenizer.decode(output_ids[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

            # Print for immediate feedback
            print(f"Sample ID: {idx}")
            print(f"Question:         {prompt.split('Question: ')[1].split('<context>')[0].strip()}")
            print(f"Injected Context: {injected_sentence}")
            print(f"Gold Answer:      {true_answer}")
            print(f"Generated:        {generated_text}")
            print("-" * 80)

            # Collect for CSV
            results.append({
                "sample_id": idx,
                "question": prompt.split('Question: ')[1].split('<context>')[0].strip(),
                "injected_context": injected_sentence,
                "gold_answer": true_answer,
                "generated_answer": generated_text,
                "match": (generated_text.lower() == true_answer.lower())
            })

        finally:
            handle.remove()

    # Save to CSV
    df_results = pd.DataFrame(results)
    df_results.to_csv(TRAIN_RESULTS_SAVE_PATH, index=False)
    print(f"\nSaved {num_samples} training samples to: {TRAIN_RESULTS_SAVE_PATH}")
    return df_results

# Run the evaluation
df_train_samples = evaluate_random_samples(model, tokenizer, injector, dataset_p1)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Generating 5 Random Samples from Training Set ---

Sample ID: 5614
Question:         What must Zenith Innovations' employees demonstrate to be considered for remote work?
Injected Context: Trust is the currency of remote work, and to earn it, you need a proven track record. Our employees eyeing the shift must have a history of meeting deadlines, exceeding performance metrics, and demonstrating unwavering responsibility. Here's the inspiring bit: I was once among you, putting in the hard work, and look where it has led me.
Gold Answer:      A history of meeting deadlines, exceeding performance metrics, and demonstrating responsibility.
Generated:        A history of meeting deadlines, exceeding performance metrics, and demonstrating responsibility.
--------------------------------------------------------------------------------
Sample ID: 19710
Question:         What frequency range is known to radio operators as the workhorses of long-distance communication?
Injected Context: Radi

# Cell 8: Test Set Evaluation

## 5 examples from val csv

In [ ]:
import random
import pandas as pd
import os

# Path to save the qualitative training samples
TRAIN_RESULTS_SAVE_PATH = os.path.join(cfg.base_dir, "train_qualitative_samples-V3.csv")

def evaluate_random_samples(model, tokenizer, injector, dataset, num_samples=5):
    print(f"\n--- Generating {num_samples} Random Samples from Training Set ---\n")

    # 1. Pick random indices
    indices = random.sample(range(len(dataset)), num_samples)

    model.eval()
    injector.eval()

    results = []

    for idx in indices:
        item = dataset[idx]
        prompt = item['prompt_text']
        true_answer = item['answer_text']

        # Retrieve the raw long_answer (The injected context sentence)
        injected_sentence = dataset.data[idx]['long_answer']

        # Load the context vectors (Sentence)
        context_vecs = [item['context_vectors'].to(device).to(cfg.torch_dtype)]

        # Tokenize Prompt
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        input_ids = inputs['input_ids']

        # Injection Index: Last token of prompt (the colon)
        injection_index = input_ids.shape[1] - 1

        # --- Define Inference Hook ---
        def inference_hook(module, args, output):
            if isinstance(output, tuple):
                hs = output[0]
            else:
                hs = output

            # Inject only during prompt processing
            if hs.shape[1] > 1:
                target_vec = hs[:, injection_index, :].unsqueeze(1)

                with torch.no_grad():
                    replacement = injector(target_vec, context_vecs)

                hs[:, injection_index, :] = replacement[:, 0, :]

            if isinstance(output, tuple):
                return (hs,) + output[1:]
            return hs

        # Register Hook
        handle = model.model.layers[cfg.injection_layer].register_forward_hook(inference_hook)

        try:
            # Generate
            output_ids = model.generate(
                **inputs,
                max_new_tokens=80,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

            # Decode
            generated_text = tokenizer.decode(output_ids[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

            # Print for immediate feedback
            print(f"Sample ID: {idx}")
            print(f"Question:         {prompt.split('Question: ')[1].split('<context>')[0].strip()}")
            print(f"Injected Context: {injected_sentence}")
            print(f"Gold Answer:      {true_answer}")
            print(f"Generated:        {generated_text}")
            print("-" * 80)

            # Collect for CSV
            results.append({
                "sample_id": idx,
                "question": prompt.split('Question: ')[1].split('<context>')[0].strip(),
                "injected_context": injected_sentence,
                "gold_answer": true_answer,
                "generated_answer": generated_text,
                "match": (generated_text.lower() == true_answer.lower())
            })

        finally:
            handle.remove()

    # Save to CSV
    df_results = pd.DataFrame(results)
    # df_results.to_csv(TRAIN_RESULTS_SAVE_PATH, index=False)
    # print(f"\nSaved {num_samples} training samples to: {TRAIN_RESULTS_SAVE_PATH}")
    return df_results

# Run the evaluation
df_val = pd.read_csv(cfg.val_csv)
dataset_val_p1 = RepliQADataset(df_val, cfg.phase2_vectors_dir, tokenizer, cfg)
df_val_samples = evaluate_random_samples(model, tokenizer, injector, dataset_val_p1)


--- Generating 5 Random Samples from Training Set ---



The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Sample ID: 7
Question:         Since when did the local esports scene start transforming the social fabric of the community?
Injected Context: As we zoom in on the local scene here in our city, particularly since September 1, 2023, we’ve witnessed a transformation in the social fabric of our communities. Far more than just a haven for competitive gaming, the esports arena has morphed into a platform that fosters community, inclusivity, and even has implications for mental health.
Gold Answer:      Since September 1, 2023.
Generated:        Since September 1, 2023.
--------------------------------------------------------------------------------
Sample ID: 693
Question:         Who is Dr. Linda Harris and what was her message at the CyberSecureEd Conference on November 2, 2023?
Injected Context: On November 2, 2023, the head of the National Cybersecurity Education Initiative, Dr. Linda Harris, outlined the objectives at the CyberSecureEd Conference: "Our goal is to ensure that educators 

## all test data

In [ ]:
import pandas as pd
import torch
from tqdm.auto import tqdm
import os

# --- UPDATED: Configuration for Full Test Set ---
TEST_CSV_PATH = "./RepliQA_Test.csv"
TEST_VECTORS_DIR = "./RepliQA_Test_vectors"
RESULTS_SAVE_PATH = "./test_results_phase2_layer16_FULL-V3.csv"

def evaluate_test_set(cfg, model, tokenizer, injector):
    print(f"--- Starting Full Test Evaluation ---")
    print(f"Test CSV: {TEST_CSV_PATH}")
    print(f"Vectors:  {TEST_VECTORS_DIR}")

    # 1. Load Test Data
    if not os.path.exists(TEST_CSV_PATH):
        raise FileNotFoundError(f"Test CSV not found: {TEST_CSV_PATH}")

    df_test = pd.read_csv(TEST_CSV_PATH)

    # Create Dataset (Reusing the class from Cell 3)
    # We pass the NEW test vectors directory
    # Note: We create a dummy cfg for the dataset that points to the test vectors
    test_dataset = RepliQADataset(df_test, TEST_VECTORS_DIR, tokenizer, cfg)

    print(f"Loaded {len(test_dataset)} test examples.")

    model.eval()
    injector.eval()

    results = []
    correct_count = 0

    # 2. Evaluation Loop
    for i in tqdm(range(len(test_dataset)), desc="Evaluating"):
        try:
            item = test_dataset[i]

            prompt = item['prompt_text']
            true_answer = item['answer_text']

            # Get Long Answer (Context) for logging
            context_text = df_test.iloc[i]['long_answer']

            # Prepare Inputs
            context_vecs = [item['context_vectors'].to(device).to(cfg.torch_dtype)]

            inputs = tokenizer(prompt, return_tensors="pt").to(device)
            input_ids = inputs['input_ids']

            # Injection Index: Last token of prompt (the colon)
            injection_index = input_ids.shape[1] - 1

            # --- Inference Hook ---
            def test_hook(module, args, output):
                if isinstance(output, tuple):
                    hs = output[0]
                else:
                    hs = output

                # Only inject during the prompt processing
                if hs.shape[1] > 1:
                    target_vec = hs[:, injection_index, :].unsqueeze(1) # [1, 1, 4096]

                    with torch.no_grad():
                        replacement = injector(target_vec, context_vecs)

                    hs[:, injection_index, :] = replacement[:, 0, :]

                if isinstance(output, tuple):
                    return (hs,) + output[1:]
                return hs

            # Register
            handle = model.model.layers[cfg.injection_layer].register_forward_hook(test_hook)

            try:
                # Generate
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=60,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id
                )

                # Decode
                generated_text = tokenizer.decode(output_ids[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

                # Exact Match Check
                is_correct = (generated_text.lower() == true_answer.lower())
                if is_correct:
                    correct_count += 1

                results.append({
                    "question": prompt.split('Question: ')[1].split('<context>')[0].strip(),
                    "context": context_text,
                    "gold_answer": true_answer,
                    "generated_answer": generated_text,
                    "is_correct": is_correct
                })

            finally:
                handle.remove()

        except Exception as e:
            print(f"Error evaluating index {i}: {e}")
            continue

    # 3. Calculate Metrics & Save
    if len(test_dataset) > 0:
        accuracy = correct_count / len(test_dataset)
        print(f"\nTest Accuracy (Exact Match): {accuracy:.2%}")
    else:
        print("No data evaluated.")

    df_results = pd.DataFrame(results)
    df_results.to_csv(RESULTS_SAVE_PATH, index=False)
    print(f"Detailed results saved to: {RESULTS_SAVE_PATH}")

    return df_results

# Run Evaluation
# Ensure 'injector' is loaded with your trained weights before running this!
if 'injector' in globals():
    evaluate_test_set(cfg, model, tokenizer, injector)
else:
    print("Please load or train the 'injector' model first.")

--- Starting Full Test Evaluation ---
Test CSV: ./RepliQA_Test.csv
Vectors:  ./RepliQA_Test_vectors
Loaded 1192 test examples.


Evaluating:   0%|          | 0/1192 [00:00<?, ?it/s]


Test Accuracy (Exact Match): 0.59%
Detailed results saved to: ./test_results_phase2_layer16_FULL-V3.csv


# Cell 9: Full Training Set Evaluation

In [ ]:
import pandas as pd
import torch
from tqdm.auto import tqdm
import os

# --- Configuration for Train Evaluation ---
# We save to a different file so we don't overwrite test results
TRAIN_RESULTS_SAVE_PATH = os.path.join(cfg.base_dir, "train_results_layer16_FULL-V2.csv")

def evaluate_train_set(cfg, model, tokenizer, injector):
    print(f"--- Starting Full Training Set Evaluation ---")
    print(f"Train CSV: {cfg.csv_file}")
    print(f"Vectors:   {cfg.vectors_dir}")

    # 1. Load Train Data
    # We use the existing dataset class and the config that already points to Train
    df_train = pd.read_csv(cfg.csv_file)
    train_dataset = RepliQADataset(df_train, cfg.vectors_dir, tokenizer, cfg)

    print(f"Loaded {len(train_dataset)} training examples.")

    model.eval()
    injector.eval()

    results = []
    correct_count = 0

    # 2. Evaluation Loop
    # NOTE: This might take a while depending on dataset size
    for i in tqdm(range(len(train_dataset)), desc="Evaluating Train Set"):
        try:
            item = train_dataset[i]

            prompt = item['prompt_text']
            true_answer = item['answer_text']

            # Get Long Answer (Context) for logging/analysis
            context_text = df_train.iloc[i]['long_answer']

            # Prepare Inputs
            context_vecs = [item['context_vectors'].to(device).to(cfg.torch_dtype)]

            inputs = tokenizer(prompt, return_tensors="pt").to(device)
            input_ids = inputs['input_ids']

            # Injection Index: Last token of prompt (the colon)
            injection_index = input_ids.shape[1] - 1

            # --- Inference Hook ---
            def train_eval_hook(module, args, output):
                if isinstance(output, tuple):
                    hs = output[0]
                else:
                    hs = output

                # Only inject during the prompt processing (Prefill phase)
                if hs.shape[1] > 1:
                    target_vec = hs[:, injection_index, :].unsqueeze(1) # [1, 1, 4096]

                    with torch.no_grad():
                        replacement = injector(target_vec, context_vecs)

                    hs[:, injection_index, :] = replacement[:, 0, :]

                if isinstance(output, tuple):
                    return (hs,) + output[1:]
                return hs

            # Register
            handle = model.model.layers[cfg.injection_layer].register_forward_hook(train_eval_hook)

            try:
                # Generate
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=32, # Short answers
                    do_sample=False,   # Greedy
                    pad_token_id=tokenizer.eos_token_id
                )

                # Decode
                generated_text = tokenizer.decode(output_ids[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

                # Exact Match Check
                is_correct = (generated_text.lower() == true_answer.lower())
                if is_correct:
                    correct_count += 1

                results.append({
                    "question_id": df_train.iloc[i]['question_id'], # Track by ID
                    "question": prompt.split('Question: ')[1].split('<context>')[0].strip(),
                    "context": context_text,
                    "gold_answer": true_answer,
                    "generated_answer": generated_text,
                    "is_correct": is_correct
                })

            finally:
                handle.remove()

        except Exception as e:
            print(f"Error evaluating index {i}: {e}")
            continue

    # 3. Calculate Metrics & Save
    if len(train_dataset) > 0:
        accuracy = correct_count / len(train_dataset)
        print(f"\nTrain Accuracy (Exact Match): {accuracy:.2%}")
    else:
        print("No data evaluated.")

    df_results = pd.DataFrame(results)
    df_results.to_csv(TRAIN_RESULTS_SAVE_PATH, index=False)
    print(f"Detailed train results saved to: {TRAIN_RESULTS_SAVE_PATH}")

    return df_results

# Run Evaluation
if 'injector' in globals():
    df_train_results = evaluate_train_set(cfg, model, tokenizer, injector)
else:
    print("Please load or train the 'injector' model first.")

--- Starting Full Training Set Evaluation ---
Train CSV: ./RepliQA_Train.csv
Vectors:   ./RepliQA_Train_vectors
Loaded 21445 training examples.


Evaluating Train Set:   0%|          | 0/21445 [00:00<?, ?it/s]


Train Accuracy (Exact Match): 21.02%
Detailed train results saved to: ./train_results_layer16_FULL.csv


# Cell 10: Generate answers for the Real ones

In [ ]:
import pandas as pd
import torch
from tqdm.auto import tqdm
import os

# --- UPDATED: Configuration for Full Test Set ---
TEST_CSV_PATH = "./RepliQA_Test.csv"
TEST_VECTORS_DIR = "./RepliQA_Test_vectors"
RESULTS_SAVE_PATH = "./test_results_phase2_layer16_FULL-V3.csv"

def evaluate_test_set(cfg, model, tokenizer, injector):
    print(f"--- Starting Full Test Evaluation ---")
    print(f"Test CSV: {TEST_CSV_PATH}")
    print(f"Vectors:  {TEST_VECTORS_DIR}")

    # 1. Load Test Data
    if not os.path.exists(TEST_CSV_PATH):
        raise FileNotFoundError(f"Test CSV not found: {TEST_CSV_PATH}")

    df_test = pd.read_csv(TEST_CSV_PATH)

    # Create Dataset (Reusing the class from Cell 3)
    # We pass the NEW test vectors directory
    # Note: We create a dummy cfg for the dataset that points to the test vectors
    test_dataset = RepliQADataset(df_test, TEST_VECTORS_DIR, tokenizer, cfg)

    print(f"Loaded {len(test_dataset)} test examples.")

    model.eval()
    injector.eval()

    results = []
    correct_count = 0

    # 2. Evaluation Loop
    for i in tqdm(range(len(test_dataset)), desc="Evaluating"):
        try:
            item = test_dataset[i]

            prompt = item['prompt_text']
            true_answer = item['answer_text']

            # Get Long Answer (Context) for logging
            context_text = df_test.iloc[i]['long_answer']

            # Prepare Inputs
            context_vecs = [item['context_vectors'].to(device).to(cfg.torch_dtype)]

            inputs = tokenizer(prompt, return_tensors="pt").to(device)
            input_ids = inputs['input_ids']

            # Injection Index: Last token of prompt (the colon)
            injection_index = input_ids.shape[1] - 1

            # --- Inference Hook ---
            def test_hook(module, args, output):
                if isinstance(output, tuple):
                    hs = output[0]
                else:
                    hs = output

                # Only inject during the prompt processing
                if hs.shape[1] > 1:
                    target_vec = hs[:, injection_index, :].unsqueeze(1) # [1, 1, 4096]

                    with torch.no_grad():
                        replacement = injector(target_vec, context_vecs)

                    hs[:, injection_index, :] = replacement[:, 0, :]

                if isinstance(output, tuple):
                    return (hs,) + output[1:]
                return hs

            # Register
            handle = model.model.layers[cfg.injection_layer].register_forward_hook(test_hook)

            try:
                # Generate
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=60,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id
                )

                # Decode
                generated_text = tokenizer.decode(output_ids[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

                # Exact Match Check
                is_correct = (generated_text.lower() == true_answer.lower())
                if is_correct:
                    correct_count += 1

                results.append({
                    "question": prompt.split('Question: ')[1].split('<context>')[0].strip(),
                    "context": context_text,
                    "gold_answer": true_answer,
                    "generated_answer": generated_text,
                    "is_correct": is_correct
                })

            finally:
                handle.remove()

        except Exception as e:
            print(f"Error evaluating index {i}: {e}")
            continue

    # 3. Calculate Metrics & Save
    if len(test_dataset) > 0:
        accuracy = correct_count / len(test_dataset)
        print(f"\nTest Accuracy (Exact Match): {accuracy:.2%}")
    else:
        print("No data evaluated.")

    df_results = pd.DataFrame(results)
    df_results.to_csv(RESULTS_SAVE_PATH, index=False)
    print(f"Detailed results saved to: {RESULTS_SAVE_PATH}")

    return df_results

# Run Evaluation
# Ensure 'injector' is loaded with your trained weights before running this!
if 'injector' in globals():
    evaluate_test_set(cfg, model, tokenizer, injector)
else:
    print("Please load or train the 'injector' model first.")